# Data retrieval

In [1]:
import pandas as pd
import numpy as np

In [2]:
# This dataset is on finance
df = pd.read_csv(r'sentiment.csv')
df

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral
...,...,...
5837,RISING costs have forced packaging producer Hu...,negative
5838,Nordic Walking was first used as a summer trai...,neutral
5839,"According shipping company Viking Line , the E...",neutral
5840,"In the building and home improvement trade , s...",neutral


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5842 entries, 0 to 5841
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Sentence   5842 non-null   object
 1   Sentiment  5842 non-null   object
dtypes: object(2)
memory usage: 91.4+ KB


# Data preprocessing

Because we're dealing with sentiments as opposed to the familiar data we used to work with, the preprocessing procedure is different. It involved:
- Deleting duplicate entries
- Text preprocessing -- which simply means turning raw text to a straightforward format the model can understand i.e without contractions, extra spaces, etc

### Duplicates

In [4]:
df.duplicated().sum()

np.int64(6)

In [5]:
df.drop_duplicates(inplace=True)

### Text preprocessing

In [6]:
import re
import contractions
import string
from nltk.tokenize import RegexpTokenizer

In [7]:
def prepped_text(text): 
    # Turn to lowercase
    text = text.lower()
    # Removing links if any
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # sub - substitutes
    # Split numbers in text
    text = re.sub(r'(\d)([A-Za-z])', r'\1 \2', text)   # number before letter
    text = re.sub(r'([A-Za-z])(\d)', r'\1 \2', text) # number after letter
    # Replace linebreaks/tabs with spaces 
    text = re.sub(r'\s+', ' ', text)
    # Replace contractions w/full words
    text = contractions.fix(text)
    # Removing punctuations 
    punctuations = string.punctuation
    for i in punctuations:
        text = text.replace(i, '')
    ## Tokenize text into a variable -- excluding only numbers 
    tokeniser = RegexpTokenizer(r'\b[a-zA-Z]{2,}\b')
    return text

In [8]:
df['cleaned_text'] = df['Sentence'].apply(prepped_text)

In [9]:
df.drop('Sentence', axis=1, inplace=True)

In [10]:
df

,Sentiment,cleaned_text
0,positive,the geosolutions technology will leverage bene...
1,negative,esi on lows down 150 to 250 bk a real possibility
2,positive,for the last quarter of 2010 componenta s net...
3,neutral,according to the finnishrussian chamber of com...
4,neutral,the swedish buyout firm has sold its remaining...
...,...,...
5837,negative,rising costs have forced packaging producer hu...
5838,neutral,nordic walking was first used as a summer trai...
5839,neutral,according shipping company viking line the eu...
5840,neutral,in the building and home improvement trade sa...


### Vectorization

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000) #

In [13]:
X = vectorizer.fit_transform(df['cleaned_text'])

### Label encoding

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
le = LabelEncoder()
df['Sentiment'] = le.fit_transform(df['Sentiment'])

# Data splitting

Before splitting the data, I checked and confirmed class imbalance. I had to try three different approaches to resolve it.
- A: This involved Oversampling followed by Undersampling using the imblearn module. The accuracy was 71%
- B: This also involved Oversampling followed by Undersampling but using imblearn and sklearn's resample function respectively. It led to an accuracy of 69% and was genuinely harder to implement
- C: This involved simply applying SMOTE technique on the minority class. It led to an accuracy of 82%. This was the easiest and most effective so I settled with it 

In [16]:
from sklearn.model_selection import train_test_split

### Class imbalance A

In [17]:
y = df['Sentiment']

In [18]:
y.value_counts()

Sentiment
1    3124
2    1852
0     860
Name: count, dtype: int64

In [19]:
y.value_counts(normalize=True)

Sentiment
1    0.535298
2    0.317341
0    0.147361
Name: proportion, dtype: float64

In [20]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [21]:
# Oversample minority class
ros = RandomOverSampler(sampling_strategy={0: 1562}, random_state=300)
X_res, y_res = ros.fit_resample(X, y)

In [22]:
y_res.value_counts()

Sentiment
1    3124
2    1852
0    1562
Name: count, dtype: int64

In [23]:
# Undersample majority class
rus = RandomUnderSampler(sampling_strategy={1: 1562, 2: 1562}, random_state=300)
X_res, y_res = rus.fit_resample(X_res, y_res)

In [24]:
y_res.value_counts()

Sentiment
0    1562
1    1562
2    1562
Name: count, dtype: int64

In [25]:
y_res.value_counts(normalize=True)

Sentiment
0    0.333333
1    0.333333
2    0.333333
Name: proportion, dtype: float64

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=123, stratify=y_res)

### Class imbalance B

In [27]:
y = df['Sentiment']

In [28]:
y.value_counts()

Sentiment
1    3124
2    1852
0     860
Name: count, dtype: int64

In [29]:
y.value_counts(normalize=True)

Sentiment
1    0.535298
2    0.317341
0    0.147361
Name: proportion, dtype: float64

##### Oversample 

In [30]:
from sklearn.utils import resample
from imblearn.over_sampling import RandomOverSampler
from scipy.sparse import vstack

In [31]:
# Oversampling minority class
os = RandomOverSampler(sampling_strategy= {0 : 1562}, random_state=26)
X_resampled, y_resampled = os.fit_resample(X, y)

In [32]:
y_resampled.value_counts()

Sentiment
1    3124
2    1852
0    1562
Name: count, dtype: int64

##### Undersample

In [33]:
y_one = y_resampled[y_resampled == 1]
one_rows = y_one.index
X_one = X_resampled[one_rows]

In [34]:
y_two = y_resampled[y_resampled == 2]
two_rows = y_two.index
X_two = X_resampled[two_rows]

In [35]:
y_zero = y_resampled[y_resampled == 0]
zero_rows = y_zero.index
X_zero = X_resampled[zero_rows]

In [36]:
X_one_res, y_one_res = resample(X_one, y_one, replace=False, n_samples=1562, random_state=42)
len(y_one_res)

1562

In [37]:
X_two_res, y_two_res = resample(X_two, y_two, replace=False, n_samples=1562, random_state=42)
len(y_two_res)

1562

##### Combine

In [38]:
# Combine X (sparse features)
X_resampled = vstack([X_one_res, X_two_res, X_zero])
# Combine y (labels)
y_resampled = np.concatenate([y_one_res, y_two_res, y_zero])

In [39]:
X_resampled

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 45855 stored elements and shape (4686, 5000)>

In [40]:
y_resampled

array([1, 1, 1, ..., 0, 0, 0], shape=(4686,))

In [41]:
# Number of rows
n_samples = X_resampled.shape[0]
# Generate shuffled indices
shuffled_idx = np.random.permutation(n_samples)
# Apply shuffle
X_resampled = X_resampled[shuffled_idx]
y_resampled = y_resampled[shuffled_idx]

##### Splitting 

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.25, random_state=42, stratify=y_resampled)

### Class imbalance C

In [43]:
y = df['Sentiment']

In [44]:
y.value_counts()

Sentiment
1    3124
2    1852
0     860
Name: count, dtype: int64

In [45]:
y.value_counts(normalize=True)

Sentiment
1    0.535298
2    0.317341
0    0.147361
Name: proportion, dtype: float64

In [46]:
from imblearn.over_sampling import SMOTE

In [47]:
sm = SMOTE(sampling_strategy={2: 3100, 0: 3100}, random_state=42)
X_res, y_res = sm.fit_resample(X, y)

In [48]:
y_res.value_counts()

Sentiment
1    3124
2    3100
0    3100
Name: count, dtype: int64

In [49]:
y_res.value_counts(normalize=True)

Sentiment
1    0.335049
2    0.332475
0    0.332475
Name: proportion, dtype: float64

In [50]:
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, random_state=300, stratify=y_res, test_size=0.2)

# Model building

In [51]:
from sklearn.svm import SVC

In [52]:
%%time
model = SVC(kernel='linear') #
model.fit(X_train, y_train)

CPU times: total: 7.27 s
Wall time: 7.59 s


,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


### Hyperparameter tuning 

Building on the applied smote technique, I used GridSearchCV Hyperparameter tuning to further improve my metrics to achieve an accuracy of 91% by playing around with the values for C until it returned the best parameters

In [53]:
from sklearn.model_selection import GridSearchCV

In [54]:
# Define parameter grid
param_grid = {
    'C': [0.01, 0.1, 1.0, 1.2, 1.5, 2, 2.5, 3, 3.5, 4, 5, 10, 100],
    'kernel': ['linear']
    # Data is linearly seperable so you don't need rbf
}

In [55]:
# GridSearchCV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',   # optimized for accuracy
    cv=5,                 # 5-fold cross-validation
    verbose=3,
    n_jobs=-1             # use all available CPU cores
)

In [56]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 13 candidates, totalling 65 fits


,estimator,SVC(kernel='linear')
,param_grid,"{'C': [0.01, 0.1, ...], 'kernel': ['linear']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,2.5


In [57]:
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'C': 2.5, 'kernel': 'linear'}


# Evaluation

In [58]:
from sklearn.metrics import classification_report

In [59]:
y_pred = grid_search.predict(X_test)

In [60]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.93      0.85       620
           1       0.82      0.68      0.74       625
           2       0.87      0.85      0.86       620

    accuracy                           0.82      1865
   macro avg       0.82      0.82      0.82      1865
weighted avg       0.82      0.82      0.82      1865



# Test

Tested the model using my own personal sentiment and it worked as intended

In [61]:
def test_case(text):
    # Clean text
    text = prepped_text(text)
    # Vectorize text
    text = vectorizer.transform([text])
    # Resulting text
    return text

In [64]:
# Fixed the test case to suit a finance sentiment analysis case
pp_text = test_case('Nasdaq share price drop by $10')
model.predict(pp_text)

array([0])